# Etapa 2 — Generación del dataset de entrenamiento

Genera `training_data/train.jsonl`, `validation.jsonl` y `test_questions.json`  
usando plantillas determinísticas + datos reales de `src/analytics.py`.  
Sin APIs, sin transformers, sin torch.

**Estructura de cada ejemplo:**
```json
{
  "intent": "ranking_accidentes",
  "input":  "<prompt completo construido por construir_prompt()>",
  "output": "<respuesta ideal en 4 secciones>"
}
```

In [ ]:
import sys
import json
import random
import re
from pathlib import Path
from collections import Counter

import pandas as pd

# Asegurar que el root del proyecto esté en el path
project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

SEED = 42
print('Python path OK. Proyecto en:', project_root)

## 1. Cargar datos procesados

In [ ]:
processed_dir = project_root / 'data' / 'processed'

atus_anual   = pd.read_csv(processed_dir / 'atus_anual.csv')
atus_detalle = pd.read_csv(processed_dir / 'atus_detalle.csv')
vmrc_ventas  = pd.read_csv(processed_dir / 'vmrc_ventas_mensual.csv')
vmrc_parque  = pd.read_csv(processed_dir / 'vmrc_parque_mensual.csv')

print('atus_anual  :', atus_anual.shape,  '| años:', sorted(atus_anual['anio'].unique()))
print('atus_detalle:', atus_detalle.shape)
print('vmrc_ventas :', vmrc_ventas.shape,  '| años:', sorted(vmrc_ventas['anio'].unique()))
print('vmrc_parque :', vmrc_parque.shape)

## 2. Generar corpus completo

In [ ]:
from src.training_data_generator import generar_corpus

corpus = generar_corpus(
    atus_anual, atus_detalle, vmrc_ventas, vmrc_parque, seed=SEED
)

print(f'Total ejemplos generados: {len(corpus)}')

## 3. Distribución por intent

In [ ]:
conteos = Counter(item['intent'] for item in corpus)

print(f'{"Intent":<45} {"N":>5}')
print('-' * 52)
for intent, n in sorted(conteos.items()):
    print(f'{intent:<45} {n:>5}')
print('-' * 52)
print(f'{"TOTAL":<45} {sum(conteos.values()):>5}')

## 4. Split estratificado (80 / 10 / 10)

In [ ]:
from src.training_data_generator import split_estratificado

train, val, test = split_estratificado(corpus, seed=SEED)

print(f'train : {len(train):>4} ejemplos')
print(f'val   : {len(val):>4} ejemplos')
print(f'test  : {len(test):>4} ejemplos')
print(f'total : {len(train)+len(val)+len(test):>4}')

# Verificar que cada intent está en los 3 splits
intents_train = set(x['intent'] for x in train)
intents_val   = set(x['intent'] for x in val)
intents_test  = set(x['intent'] for x in test)
todos = intents_train | intents_val | intents_test

print(f'\nIntents en train: {len(intents_train)} | val: {len(intents_val)} | test: {len(intents_test)}')
assert intents_val == todos, f'Faltan en val: {todos - intents_val}'
assert intents_test == todos, f'Faltan en test: {todos - intents_test}'
print('✓ Todos los intents representados en los 3 splits')

In [ ]:
# Tabla detallada por intent × split
print(f'{"Intent":<45} {"train":>6} {"val":>6} {"test":>6}')
print('-' * 65)
for intent in sorted(todos):
    nt = sum(1 for x in train if x['intent'] == intent)
    nv = sum(1 for x in val   if x['intent'] == intent)
    ns = sum(1 for x in test  if x['intent'] == intent)
    print(f'{intent:<45} {nt:>6} {nv:>6} {ns:>6}')

## 5. Guardar archivos

In [ ]:
from src.training_data_generator import guardar_jsonl, guardar_test_json
from src.utils import TRAINING_DATA_DIR

guardar_jsonl(train, TRAINING_DATA_DIR / 'train.jsonl')
guardar_jsonl(val,   TRAINING_DATA_DIR / 'validation.jsonl')
guardar_test_json(test, TRAINING_DATA_DIR / 'test_questions.json')

print('\nArchivos guardados en:', TRAINING_DATA_DIR)

## 6. Validaciones de calidad

In [ ]:
todos_items = train + val + test

# ── a) Sin placeholders sin resolver en output ──
bad_ph = [x for x in todos_items if re.search(r'\{[a-zA-Z_]+\}', x['output'])]
print(f'Outputs con placeholders sin resolver : {len(bad_ph)}  → {"OK" if not bad_ph else "ERROR"}')

# ── b) Aviso de no-causalidad en relacion_ventas_accidentes ──
rel_items = [x for x in todos_items if x['intent'] == 'relacion_ventas_accidentes_nacional']
sin_aviso = [x for x in rel_items if 'causalidad' not in x['output'].lower()]
print(f'relacion_ventas sin aviso causalidad  : {len(sin_aviso)}/{len(rel_items)}  → {"OK" if not sin_aviso else "ERROR"}')

# ── c) Longitud mínima de output ──
muy_cortos = [x for x in todos_items if len(x['output']) < 100]
print(f'Outputs con < 100 caracteres          : {len(muy_cortos)}  → {"OK" if not muy_cortos else "REVISAR"}')

# ── d) Verificar archivos en disco ──
for fname in ['train.jsonl', 'validation.jsonl', 'test_questions.json']:
    p = TRAINING_DATA_DIR / fname
    print(f'  {fname:<30} existe={p.exists()}  tamaño={p.stat().st_size:,} bytes')

print('\n✓ Validaciones completadas')

## 7. Revisión visual — 5 ejemplos aleatorios

In [ ]:
random.seed(SEED + 1)
muestra = random.sample(todos_items, 5)

sep = '=' * 72

for k, ex in enumerate(muestra, 1):
    print(sep)
    print(f'EJEMPLO {k}  |  intent: {ex["intent"]}')

    # Extraer la pregunta del usuario del prompt
    m = re.search(r'Pregunta del usuario:\s*"""\s*(.*?)\s*"""', ex['input'], re.DOTALL)
    pregunta = m.group(1).strip() if m else '(no encontrada)'
    print(f'PREGUNTA: {pregunta}')
    print()
    print('OUTPUT:')
    print(ex['output'])
    print()

## 8. Resumen final

| Archivo | Líneas | Destino |
|---|---|---|
| `train.jsonl` | ~341 | Entrenamiento fine-tuning (Etapa 4) |
| `validation.jsonl` | ~42 | Validación durante fine-tuning |
| `test_questions.json` | ~42 | Evaluación humana posterior |

**Formato de cada línea JSONL:**
```json
{"intent": "...", "input": "<prompt>", "output": "<respuesta ideal 4 secciones>"}
```

